# Weather Impact on Crop Yields Regression Analysis

## Overview

Conduct regression analysis to investigate whether different weather types have an impact on crop yields.

## Environment Setup

Ensure the required libraries are loaded.

In [ ]:
%pip install scikit-learn

In [ ]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
import numpy as np

## Load Data from Project Database

### Considerations

Load data from pre-prepared SQLite database.

For an initial investigation, will focus on a subset of the data. The two datasets (crops and yields) have a few common corresponding regions/areas:

|Crop Regions|
|------------|
|United Kingdom|
|England|
|Wales|
|Scotland|
|Northern Ireland|

An initial exploratory analysis will look at the whole of the **UK** to start with.

Likewise, one crop will be chosen to start with, **Spring Barley**.

### Exploring Weather Data

The weather data is available on a month-by-month basis, and aggregated by season and year, whereas crop data is only available as yearly yields.

Looking at the weather data at higher resolution may provide some insights into factors that may influence annual crop yields.

### Exploring Crop Data

Crop data for yields is only available as a yearly figure, since the crop is harvested once and this gives the values. Depending on the crop, when it is sown, how long it takes to grow, and when it is harvested will influence when the weather will have an affect. For instance, spring barley

### UK Weather impact on Spring Barley Yield

#### Get data from database

In [ ]:
project_db = 'data/project_db.db'

conn = sqlite3.connect(project_db)

# Query gets annual averages for monthly rainfall, maximum temperature, and sunshine hours.
query = """
	SELECT
		sb."Year" as year,
		sb."United Kingdom" AS uk_sb_yield,
		r.ann AS uk_rainfall,
		t.ann AS uk_max_temp,
		s.ann AS uk_sunshine

	FROM yields_spring_barley sb
		JOIN Rainfall r ON sb."Year" = r.year
		JOIN Tmax t ON sb."Year" = t.year
		JOIN Sunshine s ON sb."Year" = s.year
	
	WHERE r.area = 'UK'
		AND t.area = 'UK'
		AND s.area = 'UK'
		AND r.year >= 1999 AND r.year <= 2025
"""

df_annual_uk_sb = pd.read_sql(query, conn)
conn.close()
df_annual_uk_sb.head()

Delete the year column. Year is only used to ensure the correct yield data is associated with the correct weather data. Analysis will be looking for a relationship between weather and yield.

In [ ]:
del df_annual_uk_sb['year']

df_annual_uk_sb.head()

In [ ]:
df_annual_uk_sb.describe()

#### Graphing Constants

In [ ]:
sb_y_axis = 'Spring Barley Yield'

#### Scatter Plots

##### Rainfall

In [ ]:
# Define the arrays for the linear regression model.
x_uk_rain = np.array(df_annual_uk_sb['uk_rainfall']).reshape(-1, 1)
y_uk_sb_yield = np.array(df_annual_uk_sb['uk_sb_yield']).reshape(-1, 1)

# Create the model.
model = LinearRegression()

# Fit the model to the data
model.fit(x_uk_rain, y_uk_sb_yield)

r_sq = model.score(x_uk_rain, y_uk_sb_yield)
print(f"Coefficient of determination: {r_sq}")
print(f"Intercept: {model.intercept_}")
print(f"Coefficient: {model.coef_}")

# Make model predictions
y_pred = model.predict(x_uk_rain)

# Plot the arrays and results of the model prediction
plt.scatter(x_uk_rain, y_uk_sb_yield, color='blue', label='Actual Values')
plt.plot(x_uk_rain, y_pred, color='red', label='Model Fitted Line')
plt.title('Linear Regression: Yield vs Rainfall')
plt.xlabel('Annual Rainfall')
plt.ylabel(sb_y_axis)
plt.legend()
plt.show()

##### Maximum Temperature

In [ ]:
# Define the arrays for the linear regression model.
x_uk_max_temp = np.array(df_annual_uk_sb['uk_max_temp']).reshape(-1, 1)
y_uk_sb_yield = np.array(df_annual_uk_sb['uk_sb_yield']).reshape(-1, 1)

# Create the model.
model = LinearRegression()

# Fit the model to the data
model.fit(x_uk_max_temp, y_uk_sb_yield)

r_sq = model.score(x_uk_max_temp, y_uk_sb_yield)
print(f"Coefficient of determination: {r_sq}")
print(f"Intercept: {model.intercept_}")
print(f"Coefficient: {model.coef_}")

# Make model predictions
y_pred = model.predict(x_uk_max_temp)

# Plot the arrays and results of the model prediction
plt.scatter(x_uk_max_temp, y_uk_sb_yield, color='orange', label='Actual Values')
plt.plot(x_uk_max_temp, y_pred, color='red', label='Model Fitted Line')
plt.title('Linear Regression: Yield vs Rainfall')
plt.xlabel('Annual Monthly Average Max Temperature')
plt.ylabel(sb_y_axis)
plt.legend()
plt.show()

##### Sunshine

In [ ]:
# Define the arrays for the linear regression model.
x_uk_sunshine = np.array(df_annual_uk_sb['uk_sunshine']).reshape(-1, 1)
y_uk_sb_yield = np.array(df_annual_uk_sb['uk_sb_yield']).reshape(-1, 1)

# Create the model.
model = LinearRegression()

# Fit the model to the data
model.fit(x_uk_sunshine, y_uk_sb_yield)

r_sq = model.score(x_uk_sunshine, y_uk_sb_yield)
print(f"Coefficient of determination: {r_sq}")
print(f"Intercept: {model.intercept_}")
print(f"Coefficient: {model.coef_}")

# Make model predictions
y_pred = model.predict(x_uk_sunshine)

# Plot the arrays and results of the model prediction
plt.scatter(x_uk_sunshine, y_uk_sb_yield, color='magenta', label='Actual Values')
plt.plot(x_uk_sunshine, y_pred, color='red', label='Model Fitted Line')
plt.title('Linear Regression: Yield vs Sunshine')
plt.xlabel('Annual Monthly Average Sunshine')
plt.ylabel(sb_y_axis)
plt.legend()
plt.show()

#### Correlations

##### Rainfall

In [ ]:
corr = df_annual_uk_sb['uk_sb_yield'].corr(df_annual_uk_sb['uk_rainfall'])
corr

##### Maximum Temperature

In [ ]:
corr = df_annual_uk_sb['uk_sb_yield'].corr(df_annual_uk_sb['uk_max_temp'])
corr

##### Sunshine

In [ ]:
corr = df_annual_uk_sb['uk_sb_yield'].corr(df_annual_uk_sb['uk_sunshine'])
corr

### Northern Ireland Regional Analysis

#### Discussion

 The initial results looking at the influence of various weather factors for the yield of spring barley doesn't appear to show any clear correlation.
 
 The UK has a broad and varied climate, where the weather can be very different between regions. Using average values for weather across the whole of the UK to assess the impact on crop yields across the whole of the UK may hide the impact of more variable weather differences across regions.

 To try and mitigate the dilution of data from across a highly variable landscape, focusing the analysis on a single smaller region may show more interesting results. Northern Ireland is available as a region in both the weather and crops datasets. and will be used for this purpose.

#### NI data from SQLite database

In [ ]:
project_db = 'data/project_db.db'

conn = sqlite3.connect(project_db)

query = """
	SELECT
		sb."Year" as year,
		sb."Northern Ireland" AS ni_sb_yield,
		r.ann AS ni_rainfall,
		t.ann AS ni_max_temp,
		s.ann AS ni_sunshine

	FROM yields_spring_barley sb
		JOIN Rainfall r ON sb."Year" = r.year
		JOIN Tmax t ON sb."Year" = t.year
		JOIN Sunshine s ON sb."Year" = s.year
	
	WHERE r.area = 'Northern_Ireland'
		AND t.area = 'Northern_Ireland'
		AND s.area = 'Northern_Ireland'
		AND r.year >= 1999 AND r.year <= 2025
"""

df_annual_ni_sb = pd.read_sql(query, conn)
conn.close()
df_annual_ni_sb.head()

#### Scatter Plots

##### Rainfall

In [ ]:
df_annual_ni_sb.plot(x='ni_rainfall',y='ni_sb_yield', style='o')
plt.title('Yield with Rainfall')
plt.xlabel('Annual Rainfall')
plt.ylabel(sb_y_axis)
plt.show()

##### Maximum Temperature

In [ ]:
df_annual_ni_sb.plot(x='ni_max_temp',y='ni_sb_yield', style='o')
plt.title('Yield with Max Temp')
plt.xlabel('Annual Max Temp')
plt.ylabel(sb_y_axis)
plt.show()

##### Sunshine

In [ ]:
df_annual_ni_sb.plot(x='ni_sunshine',y='ni_sb_yield', style='o')
plt.title('Yield with Sunshine')
plt.xlabel('Annual Sunshine')
plt.ylabel(sb_y_axis)
plt.show()

#### Correlations

##### Rainfall

In [ ]:
corr = df_annual_ni_sb['ni_sb_yield'].corr(df_annual_ni_sb['ni_rainfall'])
corr

##### Maximum Temperature

In [ ]:
corr = df_annual_ni_sb['ni_sb_yield'].corr(df_annual_ni_sb['ni_max_temp'])
corr

##### Sunshine

In [ ]:
corr = df_annual_ni_sb['ni_sb_yield'].corr(df_annual_ni_sb['ni_sunshine'])
corr

### Narrowing the Scope

Crops are sown, or drilled at different times of year, grow over a specific period, and are harvested at a different times of the year. Yields may be reported on an annual basis, bu the time periods in the year that influence those yields will be for only the part of the year immediately preceding the sowing, as the crop grows, up to when it is harvested.

The next analyses will look at more granular weather periods, for different weather types, for different crops.